# Synergy prediction using bulk RNA-seq data

## Data loading

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import functools
from functools import reduce

Load the time-matched log2foldchange data and the CFU data.

In [26]:
from src.dge_data import get_l2fc_and_cfu_data

# Local data directories
cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"
l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge_timezero"

# Data pipeline
data_df = get_l2fc_and_cfu_data(l2fc_dir, cfu_dir, time_matched = True)

# Separate out the NDC CFU data and remove from the dataframe
cfus = data_df["CFU"]
ndc_ids = ["NDC0hr", "NDC1hr", "NDC2hr", "NDC4hr"]
ndc_cfus = [cfus.iloc[cfus.index == id].values[0] for id in ndc_ids]
ndc_cfus = ndc_cfus[1:]

# Remove the NDC columns from data df
data_df = data_df.iloc[[i for i in range(data_df.shape[0]) if data_df.index[i] not in ndc_ids]]

Now, we can construct a metadata matrix that we will bind to the data. We will eventually have to calculate synergy scores and transcriptional interaction scores for all genes, so the annotated metatdata will help us match the approriate columns to calculate these. We will implement a few functions to help us do this.

In [22]:
def condition_to_replicate_idx(condition_list):
    """
    Function to convert a list of conditions into replicate labels

    Args:
        condition_list : List of condition labels, ex: ["12CIP1hr-a", "12CIP1hr-b",...]

    Returns:
        idx_list : List of condition labels converted into replicate indices, ex: [0, 0, 1,...]
    """
    idx_list = [0]

    for i in range(1, len(condition_list)):
        prev_condition = condition_list[i-1]   
        curr_condition = condition_list[i]

        # Exclude last character since "-a" "-b"
        if prev_condition[:-1] == curr_condition[:-1]:
            idx_list.append(idx_list[i-1])
        
        else:
            idx_list.append(idx_list[i-1] + 1)
    
    return idx_list
    

def find_first_alpha(str):
    """
    Function to find the index of the the first letter in a string
    (will be used for drug name parser function)

    Args:
        str : string

    Returns:
        first_alpha : idx of first letter
    """
    for i in range(len(str)):
        if str[i].isalpha():
            return i
            break 


def condition_to_drug_id(condition_label):
    """
    Function to convert a condition label to a drug ID

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        drug_id : Drug Id as a single string, ex: "CEF", "CIP+VNC"
    """
    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        drug_id = condition_label[first_alpha_idx:first_alpha_idx + 3]
        return drug_id
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        drug1 = condition_label[first_idx:first_idx + 3]

        # Find first letter in remaining string and extract drug name
        substr     = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        drug2      = substr[second_idx:second_idx + 3]

        # Merge drug names
        drug_id = drug1 + "+" + drug2

        return drug_id

    else:
        raise KeyError("Condition label does not fit drug ID criteria")


def condition_to_drugs(condition_label):
    """
    Function to convert a condition label to a list of the two drugs represented

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        drugs : List of drugs (length 2, "" if 2nd drug doesn't exist), ex: ["CEF", NA], ["CIP", NA]
    """
    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        drug = condition_label[first_alpha_idx:first_alpha_idx + 3]
        drugs = [drug, np.nan]
        return drugs
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        drug1 = condition_label[first_idx:first_idx + 3]

        # Find first letter in remaining string and extract drug name
        substr = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        drug2 = substr[second_idx:second_idx + 3]
        drugs = [drug1, drug2]

        return drugs

    else:
        raise KeyError("Condition label does not fit drug ID criteria")
    

def condition_to_dose(condition_label):
    """
    Function to convert a condition label to a list of the two drugs represented

    Args:
        condition_label : Condition label, ex: "12CEF1hr", "12CIP13VNC2hr"

    Returns:
        doses : List of doses (length 2, "" if 2nd drug doesn't exist), ex: [0.50, NA], [0.50, 0.33]
    """
    # Define dose mapping
    dose_map = {
        "14": 0.25,
        "13": 0.33,
        "12": 0.50,
        "34": 0.75,
        "1": 1
    }

    # Count # of alpha characters
    num_upper = np.sum([char.isupper() for char in condition_label])

    # Single drug case
    if num_upper == 3:

        # Find drug name using letter search
        first_alpha_idx = find_first_alpha(condition_label)
        dose = condition_label[:first_alpha_idx]
        dose = dose_map[dose]
        doses = [dose, np.nan]
        return doses
    
    # Multiple drug case
    elif num_upper == 6:

        # Find first letter and extract drug name
        first_idx = find_first_alpha(condition_label)
        dose1 = condition_label[:first_idx]
        dose1 = dose_map[dose1]

        # Find first letter in remaining string and extract drug name
        substr = condition_label[first_idx + 3:]
        second_idx = find_first_alpha(substr)
        dose2 = substr[:second_idx]
        dose2 = dose_map[dose2]
        doses = [dose1, dose2]

        return doses

    else:
        raise KeyError("Condition label does not fit drug ID criteria")


def condition_to_timepoint(condition_label):
    """
    Function to convert a condition label to a timepoint

    Args:
        condition_label : Condition label, ex: "12CEF1hr"

    Returns:
        timepoint : Time, ex: 1
    """
    time_map = {
        "1hr": 1,
        "2hr": 2,
        "4hr": 4
    }

    # Find the timepoint label
    time_idx = condition_label.find("hr")
    timepoint = condition_label[time_idx - 1:]

    # Convert to int
    timepoint = time_map[timepoint]

    return timepoint


def condition_to_ndc_cfu(condition_label, ndc_cfus):
    """
    Function to convert a condition label to the corresponding timepoint CFU
    
    Args:
        condition_label : Condition label, ex: "12CEF1hr"
        ndc_cfus        : Array of 1hr, 2hr, 4hr NDC CFUs in order

    Returns:
        ndc_cfu : CFU value for NDC at corresponding timepoint
    """
    # Convert label to timepoint
    timepoint = condition_to_timepoint(condition_label)

    # Define map from timepoint to corresponding NDC CFU
    time_to_cfu = {
        1: ndc_cfus[0],
        2: ndc_cfus[1],
        4: ndc_cfus[2]
    }

    ndc_cfu = time_to_cfu[timepoint]

    return ndc_cfu

Now, we can apply these functions to construct a metadata matrix.

In [27]:
# Get condition ID names
labels = data_df.index

# Extract metadata from condition names
drug_id = [condition_to_drug_id(label) for label in labels]
drug = [condition_to_drugs(label) for label in labels]
drug1 = [x[0] for x in drug]
drug2 = [x[1] for x in drug]

dose = [condition_to_dose(label) for label in labels]
dose1 = [x[0] for x in dose]
dose2 = [x[1] for x in dose]

timepoint = [condition_to_timepoint(label) for label in labels]

ndc_cfu = [condition_to_ndc_cfu(label, ndc_cfus) for label in labels]

# Add metadata columns to data df
data_df["drug_id"] = drug_id
data_df["drug1"] = drug1
data_df["drug2"] = drug2
data_df["drug1_dose"] = dose1
data_df["drug2_dose"] = dose2
data_df["timepoint"] = timepoint
data_df["ndc_cfu"] = ndc_cfu
data_df["num_drugs"] = [2 if "+" in data_df["drug_id"].iloc[i] else 1 for i in range(data_df.shape[0])]
data_df["survival_fraction"] = data_df["CFU"] / data_df["ndc_cfu"] # treated CFU / NDC CFU (for Bliss score)
data_df = data_df.drop(columns = ["CFU", "ndc_cfu"])
data_df.head(n = 5)

,F01-SP0085-SP0086,F02-SP0103-SP0104,F03-SP0115-SP0116,F04-SP0116-SP0117,F05-SP0117-SP0118,F06-SP0129-SP0130,F07-SP0239-SP0240,F08-SP0256-SP0257,F09-SP0257-SP0258,F10-SP0311-SP0312,...,SP_2239,SP_2240,drug_id,drug1,drug2,drug1_dose,drug2_dose,timepoint,num_drugs,survival_fraction
12AMX1hr,0.203176,0.425140,0.421574,0.957388,0.318211,-1.596493,-0.889263,-0.828391,-0.472666,0.217035,...,-0.297218,0.162779,AMX,AMX,NaN,0.5,NaN,1,1,1.815126
12AMX2hr,0.965367,0.148895,0.385028,0.636486,0.257198,-1.249236,-1.078364,-1.086718,-1.158409,-0.353335,...,-1.057505,-1.049118,AMX,AMX,NaN,0.5,NaN,2,1,1.554896
12AMX4hr,-4.889168,0.411616,1.454771,-1.652460,1.519280,-2.430634,-0.610082,-0.623271,-2.740885,-0.228778,...,-1.004689,-0.692468,AMX,AMX,NaN,0.5,NaN,4,1,0.672222
12CEF12CIP1hr,-0.160730,1.557105,-0.188761,0.115748,-1.231779,-0.077362,1.615068,1.723700,-1.129801,0.761837,...,1.091430,0.537305,CEF+CIP,CEF,CIP,0.5,0.5,1,2,0.108403
12CEF12CIP2hr,1.485571,1.617386,0.256227,-0.743481,-0.496051,1.671432,0.851560,0.584393,-0.251858,1.756313,...,1.048523,1.009008,CEF+CIP,CEF,CIP,0.5,0.5,2,2,0.036795


We separate the single and combinaion data.

In [24]:
combo_df = data_df.iloc[(data_df["num_drugs"] == 2).to_numpy()]
single_df = data_df.iloc[(data_df["num_drugs"] == 1).to_numpy()]

print(f"# of combination datapoints : {combo_df.shape[0]}")
print(f"# of single-drug datapoints : {single_df.shape[0]}")

# of combination datapoints : 144
# of single-drug datapoints : 90


Now, for each row in the combination data, we need to be able to calculate a transcriptional interaction score for each gene, as well as a Bliss synergy score. We can use the metadata-attached dataframe as a lookup table. Let's define a transcriptional interaction score that quantifies the degree of "non-additivity" of drug combination for a gene's expression.

interaction = log2fc_{A,B} - (log2fc_A + log2fc_B)

We can also try implementing other transcriptional interaction scores later. We will also write a function to compute the EOB score.

In [ ]:
# Transcriptional interaction score methods
def simple_interaction_score(l2fc_a, l2fc_b, l2fc_ab):
    """
    Function to a compute a simple interaction score to quantify non-additivity in gene expression 
    using single drug log2foldchange and combination log2foldchange

    Args:
        l2fc_a   : Log2foldchange in condition A
        l2fc_b   : Log2foldchange in condition B
        l2fc_ab : Log2foldchange in combination

    Returns:
        score : Simple interaction score
    """
    score = l2fc_ab - l2fc_a - l2fc_b

    return score


# Synergy score methods

def conditional_round_down(num):
    """
    Round down a number to 1 only if its greater than 1 (useful for approx Bliss score)
    """
    if num > 1:
        return np.floor(num)
    else:
        return num


def eob_score(survival_frac_a, survival_frac_b, survival_frac_ab):
    """
    Function to calculate the excess over bliss (EOB) synergy score for given single drug survival and a 
    combination drug survival

    Args:
        survival_frac_a  : Fraction surviving cells in condition A
        survival_frac_b  : Fraction surviving cells in condition B
        survival_frac_ab : Fraction surviving cells in condition A+B
    """
    # If the fraction of survival exceeds 1, round down to 1
    frac_a = conditional_round_down(survival_frac_a)
    frac_b = conditional_round_down(survival_frac_b)
    frac_ab = conditional_round_down(survival_frac_ab)

    # Compute effects (inhibition)
    e_a = 1 - frac_a
    e_b = 1 - frac_b
    e_true = 1 - frac_ab

    # Expected effect
    e_ab = e_a + e_b - (e_a * e_b)

    # EOB score
    score = e_true - e_ab

    return score

def hsa_score(survival_frac_a, survival_frac_b, survival_frac_ab):
    """  
    Function to calculate the HSA synergy score for given single drug survival and a 
    combination drug survival
    """


    return score

    

We'll define a function to construct a dataframe for synergy prediction using the single drug and combination drug data.

In [33]:
def construct_synergy_df(combo_df, single_df, interaction_score_method, synergy_score_method):
    """
    Function to construct a dataframe for synergy prediction. Each condition corresponds to a drug combination,
    and each feature will be a transcriptional interaction score calculated according to specified function.
    There will be one column of synergy score as the prediction target.

    Args:
        combo_df                 : Dataframe containing combination data
        single_df                : Dataframe containing single drug data
        interaction_score_method : Function for calculating transcriptional interaction
        synergy_score_method     : Function for calculating synergy using survival fraction

    Returns:
        synergy_df : Dataframe with transcriptaional interaction score as features and EOB score as prediction target
    """
    
    # Check that combination and single drug data dfs match columns
    if set(combo_df.columns) != set(single_df.columns):
        raise KeyError("Combination data and single drug dataframes have incompatible columns")
    
    num_combos = combo_df.shape[0]
    num_single = single_df.shape[0]

    # Identify gene / transcript columns
    gene_cols = combo_df.columns[combo_df.columns.str.contains("SP", na = False)]
    
    rows = []

    for i in range(num_combos):

        # Store current combo data and specify drug 1, drug 2, doses, and timepoint
        combo_data = combo_df.iloc[i]
        combo_id = combo_df.index[i]
        drug1 = combo_data["drug1"]
        drug2 = combo_data["drug2"]
        drug1_dose = combo_data["drug1_dose"]
        drug2_dose = combo_data["drug2_dose"]
        timepoint = combo_data["timepoint"]

        # Select the two corresponding single drug datapoints from the single drug dataframe
        single_data1_bool = [
            i for i in range(num_single) if 
            single_df.iloc[i]["drug1"] == drug1 and
            single_df.iloc[i]["drug2"].isna() and 
            single_df.iloc[i]["drug1_dose"] == drug1_dose and
            single_df.iloc[i]["timepoint"] == timepoint
        ]
        single_data2_bool = [
            i for i in range(num_single) if 
            single_df.iloc[i]["drug1"].isna() and
            single_df.iloc[i]["drug2"] == drug2 and 
            single_df.iloc[i]["drug2_dose"] == drug2_dose and
            single_df.iloc[i]["timepoint"] == timepoint
        ]
        single_data1 = single_df.iloc[single_data1_bool]
        single_data2 = single_df.iloc[single_data2_bool]

        # Extract the survival fractions and compute synergy score
        surv_frac_combo = combo_data["survival_fraction"]
        surv_frac1 = single_data1["survival_fraction"]
        surv_frac2 = single_data2["survival_fraction"]
        synergy_score = synergy_score_method(surv_frac1, surv_frac2, surv_frac_combo)

        # Extract gene expression values
        l2fc_1 = single_data1[gene_cols]
        l2fc_2 = single_data2[gene_cols]
        l2fc_combo = combo_data[gene_cols]
        
        # Compute transcriptional interaction score for each gene
        interaction_scores = {
            gene: interaction_score_method(a, b, ab)
            for gene, a, b, ab in zip(
                gene_cols,
                l2fc_1,
                l2fc_2,
                l2fc_combo
            )
        }

        # Build row
        row = {
            "ID": combo_id,
            "drug1": drug1,
            "drug2": drug2,
            "drug1_dose": drug1_dose,
            "drug2_dose": drug2_dose,
            "timepoint": timepoint,
            **interaction_scores
        }
        rows.append(row)
    
    # Construct dataframe and move condition label to index
    synergy_df = pd.DataFrame(rows)
    synergy_df = synergy_df.set_index("ID")

    return synergy_df

In [ ]:
# 